# LBG REASON — Ollama éphémère (Colab / Kaggle / Lightning)

**Canonique** pour Cortex : API Ollama → `LBG_REASON_EPHEMERAL_*`.

| | |
|--|--|
| GPU | **T4** (Colab) · **P100/T4** (Kaggle) · GPU Studio (Lightning) |
| Modèle défaut | `huihui_ai/qwen3.5-abliterated:9b` (~6.5 Go) |
| Tunnel | Pinggy SSH free (alt. cloudflared : `lbg_reason_ephemeral_ollama.ipynb`) |
| Auth | Bearer obligatoire (proxy local) |
| Hors scope | Open WebUI, prod `.140` |

Doc : `packages/cortex/docs/reason_ephemeral_colab.md` · Kaggle/Lightning : section **Alternatives**.

Exécuter **dans l’ordre**. À la fin : copier le bloc `export …` en local.


In [ ]:
# 1) GPU check (Colab / Kaggle / Lightning)
# nvidia-smi -L affiche souvent « GPU 0: Tesla T4 (...) » SANS le mot « NVIDIA ».
import re, shutil, subprocess, sys

out = subprocess.getoutput("nvidia-smi -L")
print(out or "PAS DE GPU")
low = (out or "").lower()
ok = bool(
    shutil.which("nvidia-smi")
    and out
    and "failed" not in low
    and "not found" not in low
    and re.search(r"\b(tesla|t4|p100|a100|l4|v100|rtx|geforce|quadro|nvidia)\b", low)
)
if not ok:
    sys.exit(
        "Pas de GPU détecté. Colab: Runtime→T4 | Kaggle: Settings→Accelerator→GPU | Lightning: Studio GPU"
    )
print("OK GPU")


In [ ]:
# 2) Install Ollama (pas d'Open WebUI)
import pathlib, shutil, subprocess, sys

def sh(cmd: str, check: bool = True) -> subprocess.CompletedProcess:
    print("+", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.stdout:
        print(r.stdout[-1500:])
    if r.stderr and r.returncode != 0:
        print(r.stderr[-1500:])
    if check and r.returncode != 0:
        raise RuntimeError(f"échec ({r.returncode}): {cmd}")
    return r

sh("sudo apt-get update -qq && sudo apt-get install -y -qq pciutils zstd openssh-client curl >/dev/null")
if not shutil.which("ollama") and not pathlib.Path("/usr/local/bin/ollama").exists():
    sh("curl -fsSL https://ollama.com/install.sh | sh")
print("ollama:", shutil.which("ollama") or "/usr/local/bin/ollama")
sh("ollama -v", check=False)


In [ ]:
# 3) Démarrer Ollama
import os, shutil, subprocess, time

def _resolve_ollama() -> str:
    """Colab: `which` argv échoue souvent; préférer shell PATH + chemins fixes."""
    import os, shutil, subprocess
    cands: list[str] = []
    for shell_cmd in ("command -v ollama", "which ollama", "type -P ollama"):
        for line in subprocess.getoutput(shell_cmd).strip().splitlines():
            p = line.strip().split()[-1] if line.strip() else ""
            if p.startswith("/"):
                cands.append(p)
    for cand in (
        shutil.which("ollama"),
        "/usr/local/bin/ollama",
        "/usr/bin/ollama",
        "/bin/ollama",
        os.path.expanduser("~/bin/ollama"),
    ):
        if cand:
            cands.append(cand)
    for p in cands:
        if p and os.path.isfile(p):
            return p
    raise FileNotFoundError(
        "ollama introuvable — relancer la cellule d'install, puis: !command -v ollama"
    )




os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
ollama_bin = _resolve_ollama()
subprocess.run(["pkill", "-f", "ollama serve"], check=False)
time.sleep(1)
subprocess.Popen([ollama_bin, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

ok = False
for i in range(40):
    r = subprocess.run(["curl", "-fsS", "http://127.0.0.1:11434/api/tags"], capture_output=True)
    if r.returncode == 0:
        ok = True
        break
    time.sleep(1)
if not ok:
    raise RuntimeError("Ollama ne répond pas sur :11434")
print("Serveur Ollama démarré")


In [ ]:
# 4) Pull modèle abliterated T4-friendly
import os, shutil, subprocess

MODEL = os.environ.get("LBG_COLAB_MODEL", "huihui_ai/qwen3.5-abliterated:9b")
# Alts: huihui_ai/gemma-4-abliterated:12b | qwen3.5-abliterated:2B | legacy 8b-v2
# PAS: huihui_ai/Qwen3.8-abliterated:27b (heavy .110 seulement)


def _resolve_ollama() -> str:
    """Colab: `which` argv échoue souvent; préférer shell PATH + chemins fixes."""
    cands: list[str] = []
    for shell_cmd in ("command -v ollama", "which ollama", "type -P ollama"):
        for line in subprocess.getoutput(shell_cmd).strip().splitlines():
            p = line.strip().split()[-1] if line.strip() else ""
            if p.startswith("/"):
                cands.append(p)
    for cmd in (["which", "ollama"], ["bash", "-lc", "command -v ollama"]):
        try:
            out = subprocess.check_output(cmd, text=True, stderr=subprocess.PIPE).strip()
            if out:
                cands.append(out)
        except (subprocess.CalledProcessError, FileNotFoundError):
            pass
    for cand in (
        shutil.which("ollama"),
        "/usr/local/bin/ollama",
        "/usr/bin/ollama",
        "/bin/ollama",
        os.path.expanduser("~/bin/ollama"),
    ):
        if cand:
            cands.append(cand)
    for p in cands:
        # Sur Colab, os.X_OK peut mentir — exiger isfile suffit.
        if p and os.path.isfile(p):
            return p
    raise FileNotFoundError(
        "ollama introuvable — relancer la cellule d'install (curl install.sh), "
        "puis vérifier: !command -v ollama && !ollama -v"
    )


ollama_bin = _resolve_ollama()
print(f"Using Ollama binary at: {ollama_bin}")
print("Pull", MODEL, "…")
subprocess.check_call([ollama_bin, "pull", MODEL])
print(subprocess.getoutput(f'"{ollama_bin}" list'))


In [ ]:
# 5) Proxy Bearer → Ollama + tunnel Pinggy
# Ollama n'a pas d'auth native : on expose uniquement le proxy.
import json, os, re, secrets, subprocess, textwrap, threading, time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

MODEL = os.environ.get("LBG_COLAB_MODEL", "huihui_ai/qwen3.5-abliterated:9b")
LBG_TOKEN = secrets.token_urlsafe(24)
PROXY_PORT = 8088
OLLAMA = "http://127.0.0.1:11434"


def _is_tunnel_url(url: str) -> bool:
    """Évite dashboard.pinggy.io — garder tunnels *.a.* / *.free.* / etc."""
    u = url.lower().rstrip("/")
    if any(x in u for x in ("dashboard.pinggy", "www.pinggy", "docs.pinggy")):
        return False
    return bool(
        re.search(r"https?://[a-z0-9.-]+\.(a\.)?(free\.)?pinggy\.(io|link|online)(/|$)", u)
        or re.search(r"https?://[a-z0-9.-]+\.run\.pinggy", u)
    )


class Handler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        print("[proxy]", fmt % args)

    def _authorized(self) -> bool:
        return self.headers.get("Authorization", "") == f"Bearer {LBG_TOKEN}"

    def _proxy(self, method: str) -> None:
        if not self._authorized():
            self.send_response(401)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(b'{"error":"unauthorized"}')
            return
        length = int(self.headers.get("Content-Length") or 0)
        body = self.rfile.read(length) if length else None
        req = Request(OLLAMA + self.path, data=body, method=method)
        req.add_header("Content-Type", self.headers.get("Content-Type", "application/json"))
        try:
            with urlopen(req, timeout=180) as resp:
                data = resp.read()
                self.send_response(resp.status)
                self.send_header("Content-Type", resp.headers.get("Content-Type", "application/json"))
                self.send_header("Content-Length", str(len(data)))
                self.end_headers()
                self.wfile.write(data)
        except HTTPError as e:
            data = e.read()
            self.send_response(e.code)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(data)
        except URLError as e:
            self.send_response(502)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(json.dumps({"error": str(e)}).encode())

    def do_GET(self):
        self._proxy("GET")

    def do_POST(self):
        self._proxy("POST")

    def do_OPTIONS(self):
        self.send_response(204)
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Headers", "Authorization, Content-Type")
        self.send_header("Access-Control-Allow-Methods", "GET, POST, OPTIONS")
        self.end_headers()

subprocess.run(["pkill", "-f", "free.pinggy.io"], check=False)
subprocess.getoutput(f"fuser -k {PROXY_PORT}/tcp 2>/dev/null || true")
time.sleep(1)
httpd = ThreadingHTTPServer(("127.0.0.1", PROXY_PORT), Handler)
threading.Thread(target=httpd.serve_forever, daemon=True).start()
print(f"Proxy Bearer → Ollama sur :{PROXY_PORT}")

proc = subprocess.Popen(
    [
        "ssh", "-p", "443", f"-R0:localhost:{PROXY_PORT}",
        "-o", "StrictHostKeyChecking=no",
        "-o", "ServerAliveInterval=30",
        "-o", "ExitOnForwardFailure=yes",
        "free.pinggy.io",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    stdin=subprocess.DEVNULL,
    text=True,
)

PUBLIC_URL = None
deadline = time.time() + 90
buf = []
while time.time() < deadline and PUBLIC_URL is None:
    line = proc.stdout.readline()
    if not line:
        if proc.poll() is not None:
            break
        time.sleep(0.2)
        continue
    buf.append(line)
    clean = re.sub(r"\x1b\[[0-9;]*[A-Za-z]|\x1b.|\x07", "", line).strip()
    for m in re.finditer(r"https?://[^\s\"'<>]+", clean):
        cand = m.group(0).rstrip("/").rstrip(".").rstrip(",")
        if _is_tunnel_url(cand):
            PUBLIC_URL = cand
            break

if not PUBLIC_URL:
    print("".join(buf[-40:]))
    raise RuntimeError(
        "Tunnel Pinggy : URL tunnel non trouvée (évité dashboard.pinggy.io). "
        "Relancer cette cellule, ou utiliser lbg_reason_ephemeral_ollama.ipynb (cloudflared)."
    )

# Smoke /v1/models — obligatoire avant d'afficher les exports
smoke = subprocess.run(
    ["curl", "-fsS", "-H", f"Authorization: Bearer {LBG_TOKEN}", f"{PUBLIC_URL}/v1/models"],
    capture_output=True,
    text=True,
)
print("PUBLIC_URL =", PUBLIC_URL)
print("smoke /v1/models:", smoke.returncode)
print((smoke.stdout or smoke.stderr)[:600])
if smoke.returncode != 0:
    raise RuntimeError(
        f"Tunnel en apparence OK mais /v1/models échoue ({smoke.returncode}) — relancer cellule 5"
    )

block = textwrap.dedent(f'''
# --- LBG REASON ephemeral (session Colab — NE PAS push sur .140) ---
export LBG_REASON_EPHEMERAL_ENABLED=1
export LBG_REASON_EPHEMERAL_BASE_URL="{PUBLIC_URL}"
export LBG_REASON_EPHEMERAL_API_KEY="{LBG_TOKEN}"
export LBG_REASON_EPHEMERAL_MODEL="{MODEL}"
export LBG_REASON_EPHEMERAL_TIMEOUT_S=45
export LBG_REASON_EPHEMERAL_BEFORE_CLOUD=1
# Garder ce runtime Colab ouvert tant que vous testez.
''').strip()
print("\n" + "=" * 64)
print(block)
print("=" * 64)
print("\nSonde WSL :")
print('curl -sS -H "Authorization: Bearer $LBG_REASON_EPHEMERAL_API_KEY" \\')
print('  "$LBG_REASON_EPHEMERAL_BASE_URL/v1/models" | head')


## Après le run

1. Coller le bloc `export` dans votre shell WSL / poste.
2. Sonde `curl …/v1/models` puis un chat REASON si l’orchestrateur tourne en local.
3. **Disconnect runtime** dès la fin (URL + token invalidés = voulu).

Sans GPU durable : `LBG_REASON_CLOUD_*` (Groq), pas ce notebook H24.

### Modèles (override `LBG_COLAB_MODEL` avant la cellule 4)

| Tag | Note |
|-----|------|
| `huihui_ai/qwen3.5-abliterated:9b` | défaut T4 / LAN |
| `huihui_ai/gemma-4-abliterated:12b` | qualité ↑ (parité heavy) |
| `huihui_ai/qwen3.5-abliterated:2B` | smoke rapide |
| `huihui_ai/qwen3-abliterated:8b-v2` | legacy |
| `huihui_ai/qwen3-abliterated:8b-v2` | legacy |
